In [2]:
import pandas as pd 
import numpy as np 

In [3]:
results = pd.read_csv(r"dataset\results.csv")
shoots = pd.read_csv(r'dataset\shootouts.csv')

In [4]:
results.shape

(49472, 9)

In [5]:
shoots.shape

(678, 5)

In [6]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [7]:
results.dtypes

date           object
home_team      object
away_team      object
home_score    float64
away_score    float64
tournament     object
city           object
country        object
neutral          bool
dtype: object

In [8]:
results["date"] = pd.to_datetime(results['date'])

In [9]:
res = results[results['date']>'2020-01-01']
res.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
43375,2020-01-07,Barbados,Canada,1.0,4.0,Friendly,Irvine,United States,True
43376,2020-01-09,Moldova,Sweden,0.0,1.0,Friendly,Doha,Qatar,True
43377,2020-01-10,Barbados,Canada,1.0,4.0,Friendly,Irvine,United States,True
43378,2020-01-12,Kosovo,Sweden,0.0,1.0,Friendly,Doha,Qatar,True
43379,2020-01-15,Canada,Iceland,0.0,1.0,Friendly,Irvine,United States,True


In [10]:
res.isnull().sum()


date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [11]:
res = res.dropna()

In [12]:
res['home_score'] = res['home_score'].astype('int')
res['away_score'] = res['away_score'].astype('int')
res.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
43375,2020-01-07,Barbados,Canada,1,4,Friendly,Irvine,United States,True
43376,2020-01-09,Moldova,Sweden,0,1,Friendly,Doha,Qatar,True
43377,2020-01-10,Barbados,Canada,1,4,Friendly,Irvine,United States,True
43378,2020-01-12,Kosovo,Sweden,0,1,Friendly,Doha,Qatar,True
43379,2020-01-15,Canada,Iceland,0,1,Friendly,Irvine,United States,True


In [13]:
res_x = res.drop(['city','country','neutral'],axis=1)

In [14]:
res_x

,date,home_team,away_team,home_score,away_score,tournament
43375,2020-01-07,Barbados,Canada,1,4,Friendly
43376,2020-01-09,Moldova,Sweden,0,1,Friendly
43377,2020-01-10,Barbados,Canada,1,4,Friendly
43378,2020-01-12,Kosovo,Sweden,0,1,Friendly
43379,2020-01-15,Canada,Iceland,0,1,Friendly
...,...,...,...,...,...,...
49395,2026-06-09,Estonia,Lithuania,1,0,Baltic Cup
49396,2026-06-09,Russia,Trinidad and Tobago,3,0,Friendly
49397,2026-06-09,Togo,Benin,5,1,Friendly
49398,2026-06-09,Liberia,Sierra Leone,3,1,Friendly


In [15]:
#Encoding the home and away teams with their FIFA points from fifa_rankings.xlsx
fifa_rankings = pd.read_excel(r'dataset\fifa_rankings.xlsx')
fifa_rankings.head()

,Team names,FIFA points
0,France,1877
1,Spain,1876
2,Argentina,1875
3,England,1826
4,Portugal,1764


In [16]:

# Create a dictionary mapping team names to their FIFA points for faster lookup
points_mapping = dict(zip(fifa_rankings['Team names'], fifa_rankings['FIFA points']))
inverse_points_mapping = {v: k for k, v in points_mapping.items()}

def encode_teams_to_points(df, home_col='home_team', away_col='away_team'):
    df[f'{home_col}_points'] = df[home_col].map(points_mapping)
    df[f'{away_col}_points'] = df[away_col].map(points_mapping)      
    return df




In [17]:
res_x = encode_teams_to_points(res_x)

In [18]:
res_x.iloc[:6,:]

,date,home_team,away_team,home_score,away_score,tournament,home_team_points,away_team_points
43375,2020-01-07,Barbados,Canada,1,4,Friendly,910.0,1556.0
43376,2020-01-09,Moldova,Sweden,0,1,Friendly,1004.0,1515.0
43377,2020-01-10,Barbados,Canada,1,4,Friendly,910.0,1556.0
43378,2020-01-12,Kosovo,Sweden,0,1,Friendly,1319.0,1515.0
43379,2020-01-15,Canada,Iceland,0,1,Friendly,1556.0,1345.0
43380,2020-01-19,El Salvador,Iceland,0,1,Friendly,1225.0,1345.0


In [19]:
res_x.columns


Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'home_team_points', 'away_team_points'],
      dtype='object')

In [20]:
predictors = res_x.drop(['date', 'home_team', 'away_team', 'home_score', 'away_score'], axis=1)
y = res_x.iloc[:,3:5].values

In [21]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

predictors_model = predictors.copy()
le = LabelEncoder()
le.fit(res_x['tournament'])

if predictors_model['tournament'].dtype == object or str(predictors_model['tournament'].dtype).startswith('string'):
    predictors_model['tournament'] = le.transform(predictors_model['tournament'])

default_points = float(fifa_rankings['FIFA points'].median())
predictors_model = predictors_model.fillna(default_points)

X_train, X_test, y_train, y_test = train_test_split(
    predictors_model,
    y,
    test_size=0.2,
    random_state=42,
)

rf = RandomForestRegressor(
    n_estimators=200,
    min_samples_split=10,
    random_state=42,
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")





#xgboost
import xgboost as xgb
xg_reg = xgb.XGBRegressor(objective ='count:poisson', colsample_bytree = 0.3, learning_rate = 0.1,
                max_depth = 5, n_estimators = 500)
xg_reg.fit(X_train,y_train)
y_pred = xg_reg.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error(xgb): {mse}")



Mean Squared Error: 1.6629112894775118
Mean Squared Error(xgb): 1.6062073707580566


In [22]:
def predict_match_score(home_team, away_team, tournament='Friendly'):
    tournament_value = tournament if tournament in le.classes_ else le.classes_[0]
    match_input = pd.DataFrame([
        {
            'tournament': le.transform([tournament_value])[0],
            'home_team_points': float(points_mapping.get(home_team, default_points)),
            'away_team_points': float(points_mapping.get(away_team, default_points)),
        }
    ])
    match_input = match_input[predictors_model.columns]
    predicted_home_score, predicted_away_score = xg_reg.predict(match_input)[0]
    home_score = max(0, int(round(predicted_home_score)))
    away_score = max(0, int(round(predicted_away_score)))
    return f"{home_team} {home_score} - {away_score} {away_team}"

print(predict_match_score('Argentina', 'Iceland', 'Friendly'))
print(predict_match_score('Portugal', 'Nigeria', 'Friendly'))
print(predict_match_score('Brazil', 'Egypt', 'Friendly'))

Argentina 3 - 0 Iceland
Portugal 2 - 1 Nigeria
Brazil 2 - 1 Egypt


In [23]:
def predict_match_result(home_team, away_team, tournament):
    tournament_value = tournament if tournament in le.classes_ else le.classes_[0]
    match_input = pd.DataFrame([
        {
            'tournament': le.transform([tournament_value])[0],
            'home_team_points': float(points_mapping.get(home_team, default_points)),
            'away_team_points': float(points_mapping.get(away_team, default_points)),
        }
    ])
    match_input = match_input[predictors_model.columns]
    predicted_home_score, predicted_away_score = rf.predict(match_input)[0]
    home_score = max(0, int(round(predicted_home_score)))
    away_score = max(0, int(round(predicted_away_score)))
    if home_score > away_score:
        result = f"{home_team} Wins over {away_team}"
    elif away_score > home_score:
        result = f"{away_team} Wins over {home_team}"
    else:
        result = f"{home_team} and {away_team} Draw"
    return result


In [25]:
import joblib
joblib.dump(xg_reg, 'fifa_score_predictor_xgb.pkl')


['fifa_score_predictor_xgb.pkl']

In [ ]:
xg_reg = joblib.load('fifa_score_predictor_xgb.pkl')
predict_match_result('Portugal', 'Argentina', 'World Cup')


'Portugal and Argentina Draw'